In [15]:
import pandas as pd

In [16]:
tracking = pd.concat([pd.read_parquet("data/nhl_event_data_20232024.parquet"),
                      pd.read_parquet("data/nhl_event_data_20242025.parquet"),
                      pd.read_parquet("data/nhl_event_data_20252026.parquet")])
pbp = pd.concat([pd.read_csv("data/pbp_20232024.csv"),pd.read_csv("data/pbp_20242025.csv"),pd.read_csv("data/pbp_20252026.csv")])
merged = pd.merge(tracking, pbp[["gameId","eventId","xG", "periodNumber","secondsElapsedInPeriod","isHome","eventOwnerTeamId"]], left_on=["game_id","ev_id"],right_on=["gameId","eventId"])

/var/folders/7y/fwm3j5vs3p72s234bsqbz8sw0000gp/T/ipykernel_50923/4130013881.py:4: DtypeWarning: Columns (16: homeIsEmptyNet, 17: awayIsEmptyNet) have mixed types. Specify dtype option on import or set low_memory=False.
  pbp = pd.concat([pd.read_csv("data/pbp_20232024.csv"),pd.read_csv("data/pbp_20242025.csv"),pd.read_csv("data/pbp_20252026.csv")])


In [17]:
import re
import numpy as np
import pandas as pd



def normalize_direction(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()


    slot_re = re.compile(r"^(home|away)_on_ice_(\d+)_")

    # true on-ice counts, the dataset rejects frames with more than 6 and the slots that show it are dropped below
    for side in ["home", "away"]:
        side_ids = [c for c in df.columns if re.fullmatch(rf"{side}_on_ice_\d+_id", c)]
        df[f"{side}_n_on_ice"] = df[side_ids].notna().sum(axis=1)

    drop = [c for c in df.columns if (m := slot_re.match(c)) and int(m.group(2)) >= 7]
    df = df.drop(columns=drop)



    df = df.sort_values(["game_id", "ev_id", "time_stamp"])
    attack_dir = np.sign(df.groupby(["game_id", "ev_id"])["puck_x"].transform("last"))
    df["attack_dir"] = attack_dir

    x_cols = [c for c in df.columns if c.endswith("_x")]
    y_cols = [c for c in df.columns if c.endswith("_y")]
    df[x_cols] = df[x_cols].mul(attack_dir, axis=0)
    df[y_cols] = df[y_cols].mul(attack_dir, axis=0)
    return df

merged = normalize_direction(merged)

In [18]:
GOAL_LINE_X, NET_HALF_WIDTH, NET_DEPTH = 89, 3, 3.5   # goal line 11 ft in from the end boards (x=100); net ~6 ft wide, ~3.5 ft deep

merged = merged.sort_values(["game_id", "ev_id", "time_stamp"]).reset_index(drop=True)

# goal frame = first frame the puck is inside the net (past the goal line, between the posts, not behind the net)
in_net = (merged["puck_x"].between(GOAL_LINE_X, GOAL_LINE_X + NET_DEPTH)
          & (merged["puck_y"].abs() <= NET_HALF_WIDTH))
merged["goal_ts"] = merged["time_stamp"].where(in_net).groupby([merged["game_id"], merged["ev_id"]]).transform("min")

print("clips where the puck never enters the net (kept whole):",
      merged.loc[merged["goal_ts"].isna(), ["game_id", "ev_id"]].drop_duplicates().shape[0])
merged = merged[merged["goal_ts"].isna() | (merged["time_stamp"] <= merged["goal_ts"])].reset_index(drop=True)

# line changes: frames where a team has more than 6 on ice (outgoing and incoming players overlap)
# and frames where the set of players on ice differs from the previous frame of the clip
id_cols = [f"{side}_on_ice_{n}_id" for side in ["home", "away"] for n in range(1, 7)]
ids = np.sort(merged[id_cols].to_numpy(dtype=np.float64), axis=1)
prev = np.roll(ids, 1, axis=0)
clip_keys = merged[["game_id", "ev_id"]]
same_clip = (clip_keys == clip_keys.shift()).all(axis=1).to_numpy()
same_lineup = ((ids == prev) | (np.isnan(ids) & np.isnan(prev))).all(axis=1)
changing = (merged["home_n_on_ice"] > 6) | (merged["away_n_on_ice"] > 6) | (same_clip & ~same_lineup)
print("line change frames dropped:", int(changing.sum()))
merged = merged[~changing].reset_index(drop=True)


clips where the puck never enters the net (kept whole): 567
line change frames dropped: 352650


In [19]:
merged["shot_id"] = merged["game_id"].astype(str) + "_" + merged["ev_id"].astype(str)  # one sequence per goal clip
merged["is_goal"] = 1

metadata = pd.read_csv("data/metadata.csv")
pos = metadata.set_index("playerId")["position"]

slot_cols = {}
for side in ["home", "away"]:
    for n in range(1, 7):
        slot = f"{side}_on_ice_{n}"
        slot_cols[f"{slot}_pos"] = merged[f"{slot}_id"].map(pos).fillna("G").where(merged[f"{slot}_id"].notna())
        slot_cols[f"{slot}_off"] = merged[f"{slot}_team_id"] == merged["eventOwnerTeamId"]
merged = pd.concat([merged, pd.DataFrame(slot_cols, index=merged.index)], axis=1)

In [20]:
merged.to_csv("data/processed_ppt.csv")

In [21]:
merged

,time_stamp,home_on_ice_1_tracking_id,home_on_ice_1_id,home_on_ice_1_x,home_on_ice_1_y,home_on_ice_1_sweater_number,home_on_ice_1_team_id,home_on_ice_1_team_abbr,home_on_ice_2_tracking_id,home_on_ice_2_id,...,away_on_ice_2_pos,away_on_ice_2_off,away_on_ice_3_pos,away_on_ice_3_off,away_on_ice_4_pos,away_on_ice_4_off,away_on_ice_5_pos,away_on_ice_5_off,away_on_ice_6_pos,away_on_ice_6_off
0,16969742481,14021.0,8478010.0,12.460767,-33.398136,21.0,14.0,TBL,14031.0,8477992.0,...,L,False,D,False,G,False,C,False,C,False
1,16969742482,14021.0,8478010.0,14.945292,-33.006301,21.0,14.0,TBL,14031.0,8477992.0,...,L,False,D,False,G,False,C,False,C,False
2,16969742483,14021.0,8478010.0,17.402075,-32.325382,21.0,14.0,TBL,14031.0,8477992.0,...,L,False,D,False,G,False,C,False,C,False
3,16969742484,14021.0,8478010.0,19.815875,-31.389153,21.0,14.0,TBL,14031.0,8477992.0,...,L,False,D,False,G,False,C,False,C,False
4,16969742485,14021.0,8478010.0,22.163325,-30.245526,21.0,14.0,TBL,14031.0,8477992.0,...,L,False,D,False,G,False,C,False,C,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1659238,17814921149,54009.0,8478403.0,12.030492,37.123018,9.0,54.0,VGK,54016.0,8481604.0,...,C,True,D,True,L,True,G,True,R,True
1659239,17814921150,54009.0,8478403.0,13.666858,36.524839,9.0,54.0,VGK,54016.0,8481604.0,...,C,True,D,True,L,True,G,True,R,True
1659240,17814921151,54009.0,8478403.0,15.350583,35.811456,9.0,54.0,VGK,54016.0,8481604.0,...,C,True,D,True,L,True,G,True,R,True
1659241,17814921152,54009.0,8478403.0,17.085925,34.981248,9.0,54.0,VGK,54016.0,8481604.0,...,C,True,D,True,L,True,G,True,R,True
